# 03 - SHAP Sadar-Kondisi (Paper 4 XAI)

Dataset: **CIC-IDS2017 (ref) -> CSE-CIC-IDS2018 (tgt)**. SHAP dievaluasi pada target (2018).

Mengisi Tabel 7 + Seksi 3.7 paper. Menghitung SHAP (TreeSHAP) global untuk **KEDUA model**
(XGBoost & Random Forest) pada:
- **Model gagal** (transfer langsung 2017 -> 2018, recall Attack rendah)
- **Model pulih** (few-shot 1% target 2018)

lalu membandingkan peringkat fitur (stabilitas + korelasi Spearman). Tujuannya menegaskan:
stabilitas fitur SHAP tidak menjamin transferabilitas.

## Dijalankan di AWS SageMaker
> Jalankan 01 dan 02 lebih dulu. Butuh paket `shap`.


In [ ]:
# === Instal paket (SageMaker) ===
import sys
!{sys.executable} -m pip install -q numpy pandas scipy scikit-learn xgboost shap
print('deps OK')


In [ ]:
import json, pickle, os, subprocess, warnings
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

ART='./artifacts'; SEED=42; TOPK=20
S3_ARTIFACTS_OUT = 's3://ssh-detection-features-232032302717/paper4-xai/artifacts/'  # samakan dgn notebook 01
MODELS=['xgb','rf']
# === Pastikan artefak input ada; kalau belum, tarik dari S3 (mis. 01 dijalankan di sesi lain) ===
import glob
def ensure_artifacts(needed):
    os.makedirs(ART, exist_ok=True)
    missing = [f for f in needed if not os.path.exists(os.path.join(ART, f))]
    if missing:
        print('artefak lokal belum lengkap:', missing, '-> sync dari', S3_ARTIFACTS_OUT)
        subprocess.run(['aws','s3','sync',S3_ARTIFACTS_OUT,ART,'--only-show-errors'], check=True)
    else:
        print('artefak input sudah ada di', ART)
ensure_artifacts(['cic2017.pkl','cic2018.pkl','diagnosis_results.json'])

ref=pickle.load(open(f'{ART}/cic2017.pkl','rb'))   # CIC-IDS2017 (referensi/latih)
tgt=pickle.load(open(f'{ART}/cic2018.pkl','rb'))   # CSE-CIC-IDS2018 (target transfer)
FEATURES=ref['features']

def make_model(kind, seed=SEED):
    if kind=='xgb':
        from xgboost import XGBClassifier
        return XGBClassifier(max_depth=8,learning_rate=0.1,n_estimators=200,subsample=0.8,
                             colsample_bytree=0.8,objective='binary:logistic',eval_metric='logloss',
                             tree_method='hist',random_state=seed,n_jobs=-1)
    from sklearn.ensemble import RandomForestClassifier
    return RandomForestClassifier(n_estimators=200,random_state=seed,n_jobs=-1)


In [ ]:
# Util membangun model (gagal / pulih) untuk kind tertentu
def build(kind, Xsrc, ysrc, Xextra=None, yextra=None):
    X = Xsrc if Xextra is None else pd.concat([Xsrc,Xextra],ignore_index=True)
    y = ysrc if yextra is None else np.concatenate([ysrc,yextra])
    sc=StandardScaler().fit(X.values); Xs=sc.transform(X.values)
    m0=make_model(kind); m0.fit(Xs,y)
    imp=getattr(m0,'feature_importances_',np.ones(Xs.shape[1]))
    idx=np.argsort(imp)[::-1][:min(TOPK,Xs.shape[1])]
    m=make_model(kind); m.fit(Xs[:,idx],y)
    return sc,idx,m

# ref=2017 (sumber), tgt=2018 (target). SHAP dievaluasi di test target (2018).
Xref,yref=ref['X'],ref['y']; Xtgt,ytgt=tgt['X'],tgt['y']
Xr_tr,Xr_te,yr_tr,yr_te=train_test_split(Xref,yref,test_size=0.2,stratify=yref,random_state=SEED)
Xt_tr,Xt_te,yt_tr,yt_te=train_test_split(Xtgt,ytgt,test_size=0.2,stratify=ytgt,random_state=SEED)
n=max(1,int(0.01*len(yt_tr))); sel=np.random.RandomState(SEED).choice(len(yt_tr),n,replace=False)
print('data split siap')


In [ ]:
import shap, numpy as np

def _fix_xgb_base_score(model):
    # Bug kompat: xgboost baru menyimpan base_score sbg '[0.151..]' -> shap gagal float().
    # Set ulang base_score booster ke skalar agar TreeExplainer bisa parse.
    try:
        booster = model.get_booster()
        cfg = json.loads(booster.save_config())
        lp = cfg['learner']['learner_model_param']
        bs = lp.get('base_score', '0.5')
        bs = str(bs).strip().lstrip('[').rstrip(']')
        val = float(bs.split(',')[0])
        lp['base_score'] = repr(val)
        booster.load_config(json.dumps(cfg))
        return booster
    except Exception as e:
        print('   (base_score fix dilewati:', e, ')')
        return None

def _mean_abs_shap(sv, n_features):
    # Normalkan bentuk sv (list / 2D / 3D) -> vektor mean|SHAP| sepanjang n_features.
    if isinstance(sv, list):
        sv = sv[1] if len(sv) > 1 else sv[0]
    sv = np.asarray(sv)
    if sv.ndim == 3:            # (n, features, classes) -> ambil kelas positif
        sv = sv[:, :, -1]
    return np.abs(sv).mean(axis=0)[:n_features]

def shap_rank(model, scaler, idx, Xeval, nmax=2000):
    Xs = scaler.transform(Xeval.values)[:, idx]
    if len(Xs) > nmax:
        r = np.random.RandomState(SEED).choice(len(Xs), nmax, replace=False); Xs = Xs[r]
    names = [FEATURES[i] for i in idx]
    method = 'tree_shap'
    mean_abs = None
    # 1) TreeExplainer langsung
    try:
        expl = shap.TreeExplainer(model); sv = expl.shap_values(Xs)
        mean_abs = _mean_abs_shap(sv, len(idx))
    except Exception as e1:
        print('   TreeExplainer(model) gagal:', str(e1)[:80])
        # 2) perbaiki base_score lalu jelaskan booster
        try:
            booster = _fix_xgb_base_score(model)
            if booster is not None:
                expl = shap.TreeExplainer(booster); sv = expl.shap_values(Xs)
                mean_abs = _mean_abs_shap(sv, len(idx))
        except Exception as e2:
            print('   TreeExplainer(booster) gagal:', str(e2)[:80])
    # 3) fallback terakhir: feature_importances_ (BUKAN SHAP, ditandai jujur)
    if mean_abs is None:
        imp = getattr(model, 'feature_importances_', np.ones(len(idx)))
        mean_abs = np.asarray(imp)[:len(idx)]
        method = 'feature_importance_fallback'
        print('   -> fallback ke feature_importances_ (shap gagal)')
    order = np.argsort(mean_abs)[::-1]
    ranking = {names[i]: float(mean_abs[i]) for i in order}
    return ranking, method

def spearman_rank(ra, rb):
    common = [f for f in ra if f in rb]
    xa = [list(ra).index(f) for f in common]; xb = [list(rb).index(f) for f in common]
    rho, p = spearmanr(xa, xb); return round(float(rho), 4), round(float(p), 4)


In [ ]:
# Hitung SHAP untuk kedua model: model gagal vs pulih (dievaluasi di target 2018)
out = {}
for kind in MODELS:
    print(f'=== {kind} ===')
    sc0, idx0, m_fail = build(kind, Xr_tr, yr_tr)                               # gagal (2017 saja)
    sc1, idx1, m_heal = build(kind, Xr_tr, yr_tr, Xt_tr.iloc[sel], yt_tr[sel])  # pulih (+1% target 2018)
    rank_fail, meth_fail = shap_rank(m_fail, sc0, idx0, Xt_te)
    rank_heal, meth_heal = shap_rank(m_heal, sc1, idx1, Xt_te)
    rho, p = spearman_rank(rank_fail, rank_heal)
    out[kind] = {'spearman_fail_vs_heal': {'rho': rho, 'p': p},
                 'method_fail': meth_fail, 'method_heal': meth_heal,
                 'rank_fail': rank_fail, 'rank_heal': rank_heal}
    print(f'[{kind}] method fail/heal: {meth_fail} / {meth_heal}')
    print(f'[{kind}] top fail:', list(rank_fail)[:8])
    print(f'[{kind}] top heal:', list(rank_heal)[:8])
    print(f'[{kind}] Spearman gagal-vs-pulih rho={rho} (p={p})\n')

if set(MODELS) == {'xgb', 'rf'}:
    rho_xr, p_xr = spearman_rank(out['xgb']['rank_fail'], out['rf']['rank_fail'])
    out['xgb_vs_rf_fail_spearman'] = {'rho': rho_xr, 'p': p_xr}
    print('xgb vs rf (model gagal) Spearman rho=', rho_xr, '(p=', p_xr, ')')

out['ref_dataset'] = 'CIC-IDS2017'; out['tgt_dataset'] = 'CSE-CIC-IDS2018'
json.dump(out, open(f'{ART}/shap_results.json', 'w'), indent=2)
print('\nTersimpan shap_results.json')


In [ ]:
# === Upload artefak ke S3 ===
subprocess.run(['aws', 's3', 'sync', ART, S3_ARTIFACTS_OUT, '--only-show-errors'], check=True)
print('artifacts ter-upload ke', S3_ARTIFACTS_OUT)
